# Aidoc Algorithm Evaluation

Exploratory analysis notebook. Each section runs a query, displays a table, and renders a chart.

In [1]:
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

DATA_FILE = "AI_data_analysis_exercise_(4)_(2)_(2)_(4).xlsx"
COLOR_MAP = {
    "Radiologist": "#2ca02c",
    "Algo 1": "#d62728",
    "Algo 2": "#1f77b4",
    "Algo 3": "#ff7f0e",
}
ALGO_COLS = ["Algo 1", "Algo 2", "Algo 3"]
AGE_LINE_COLS = ["Radiologist", "Algo 1", "Algo 2", "Algo 3"]

df = pd.read_excel(DATA_FILE)
for col in [
    "scan_timestamp", "radiologist_sign_time", "algos_start_run",
    "algo1_finish_run", "algo2_finish_run", "algo3_finish_run",
]:
    df[col] = pd.to_datetime(df[col])

filtered_df = df.copy()
print(f"Loaded {len(filtered_df):,} scans")
print("Sites:", sorted(filtered_df["site"].unique()))
print("Departments:", sorted(filtered_df["patient_class"].unique()))
print("Gender:", sorted(filtered_df["gender"].unique()))


Loaded 20,000 scans
Sites: ['best_doctors', 'healthy_vibes']
Departments: ['ED', 'IN']
Gender: ['female', 'male']


In [2]:
def group_accuracy_query(group_col: str) -> str:
    return f"""
    SELECT
        {group_col},
        COUNT(*) AS scans,
        SUM(CASE WHEN radiologist_answer = 'P' THEN 1 ELSE 0 END) AS positives,
        ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS prevalence_pct,
        ROUND(AVG(CASE WHEN algo1_answer = radiologist_answer THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Algo 1",
        ROUND(AVG(CASE WHEN algo2_answer = radiologist_answer THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Algo 2",
        ROUND(AVG(CASE WHEN algo3_answer = radiologist_answer THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Algo 3"
    FROM filtered_df
    GROUP BY {group_col}
    ORDER BY {group_col}
    """


def accuracy_bar(df: pd.DataFrame, group_col: str, title: str) -> go.Figure:
    long = df.melt(id_vars=group_col, value_vars=ALGO_COLS, var_name="Algorithm", value_name="Accuracy (%)")
    fig = px.bar(
        long, y=group_col, x="Accuracy (%)", color="Algorithm", barmode="group", orientation="h",
        title=title, color_discrete_map=COLOR_MAP,
        labels={group_col: group_col.replace("_", " ").title()},
    )
    fig.update_layout(legend_title_text="", yaxis_title="")
    return fig


def age_subgroup_chart(age_df: pd.DataFrame) -> go.Figure:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Bar(
            x=age_df["age_group"], y=age_df["scans"], name="Scan count",
            marker_color="rgba(128, 128, 128, 0.25)",
            hovertemplate="Age %{x}<br>Scans: %{y:,}<extra></extra>",
        ),
        secondary_y=True,
    )
    for col in AGE_LINE_COLS:
        fig.add_trace(
            go.Scatter(
                x=age_df["age_group"], y=age_df[col], name=col, mode="lines+markers",
                line=dict(color=COLOR_MAP[col]), marker=dict(color=COLOR_MAP[col]),
                hovertemplate=f"{col}<br>%{{x}}: %{{y:.1f}}%<extra></extra>",
            ),
            secondary_y=False,
        )
    fig.update_layout(
        title="Accuracy & Radiologist Prevalence by Age Group",
        legend_title_text="", xaxis=dict(type="category", title="Age Group"), barmode="overlay",
    )
    fig.update_yaxes(title_text="Rate (%)", secondary_y=False, range=[0, 100])
    fig.update_yaxes(title_text="Scan count", secondary_y=True, showgrid=False)
    return fig


AGE_QUERY = """
WITH binned AS (
    SELECT *,
        CASE
            WHEN age < 3 THEN '00-03' WHEN age < 6 THEN '03-06' WHEN age < 9 THEN '06-09'
            WHEN age < 12 THEN '09-12' WHEN age < 15 THEN '12-15' WHEN age < 18 THEN '15-18'
            WHEN age < 21 THEN '18-21' WHEN age < 24 THEN '21-24' WHEN age < 27 THEN '24-27'
            WHEN age < 30 THEN '27-30' WHEN age < 32 THEN '30-32' WHEN age < 37 THEN '32-37'
            WHEN age < 42 THEN '37-42' WHEN age < 47 THEN '42-47' WHEN age < 52 THEN '47-52'
            WHEN age < 57 THEN '52-57' WHEN age < 62 THEN '57-62' WHEN age < 67 THEN '62-67'
            WHEN age < 70 THEN '67-70' WHEN age < 73 THEN '70-73' WHEN age < 76 THEN '73-76'
            WHEN age < 79 THEN '76-79' WHEN age < 82 THEN '79-82' ELSE '82+'
        END AS age_group,
        CASE WHEN algo1_answer = radiologist_answer THEN 1 ELSE 0 END AS a1_corr,
        CASE WHEN algo2_answer = radiologist_answer THEN 1 ELSE 0 END AS a2_corr,
        CASE WHEN algo3_answer = radiologist_answer THEN 1 ELSE 0 END AS a3_corr
    FROM filtered_df
)
SELECT
    age_group,
    COUNT(*) AS scans,
    ROUND(AVG(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Radiologist",
    ROUND(AVG(a1_corr) * 100, 1) AS "Algo 1",
    ROUND(AVG(a2_corr) * 100, 1) AS "Algo 2",
    ROUND(AVG(a3_corr) * 100, 1) AS "Algo 3"
FROM binned
GROUP BY age_group
ORDER BY age_group
"""


---
## 1. Patient Distribution

Scan volume across sites (`best_doctors`, `healthy_vibes`), departments (`ED`, `IN`), gender, and age.

In [3]:
# 1a. site × patient_class × gender
dist_query = """
SELECT
    site,
    patient_class,
    gender,
    COUNT(*) AS scans,
    SUM(CASE WHEN radiologist_answer = 'P' THEN 1 ELSE 0 END) AS positives,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS prevalence_pct
FROM filtered_df
GROUP BY site, patient_class, gender
ORDER BY site, patient_class, gender
"""
dist_df = duckdb.sql(dist_query).df()
display(dist_df)

fig_sun = px.sunburst(
    dist_df, path=["site", "patient_class", "gender"], values="scans",
    title="Scan Volume: Site → Department → Gender",
    color="prevalence_pct", color_continuous_scale="YlOrRd",
)
fig_sun.show()


,site,patient_class,gender,scans,positives,prevalence_pct
0,best_doctors,ED,female,602,34.0,5.6
1,best_doctors,ED,male,1047,58.0,5.5
2,best_doctors,IN,female,2221,254.0,11.4
3,best_doctors,IN,male,2116,251.0,11.9
4,healthy_vibes,ED,female,1377,67.0,4.9
5,healthy_vibes,ED,male,2502,102.0,4.1
6,healthy_vibes,IN,female,5020,543.0,10.8
7,healthy_vibes,IN,male,5115,595.0,11.6


In [4]:
# 1b. department × gender
fig_dept = px.bar(
    dist_df.groupby(["patient_class", "gender"], as_index=False)["scans"].sum(),
    x="patient_class", y="scans", color="gender", barmode="group",
    title="Scans by Department and Gender",
    labels={"patient_class": "Department (ED / IN)", "scans": "Scan Count"},
    color_discrete_map={"male": "#4c78a8", "female": "#f58518"},
)
fig_dept.update_layout(legend_title_text="")
fig_dept.show()


In [5]:
# 1c. age histogram by gender
fig_age_dist = px.histogram(
    filtered_df, x="age", color="gender", barmode="overlay", nbins=30, opacity=0.7,
    title="Age Distribution by Gender",
    labels={"age": "Age (years)"},
    color_discrete_map={"male": "#4c78a8", "female": "#f58518"},
)
fig_age_dist.update_layout(legend_title_text="")
fig_age_dist.show()


In [6]:
# 1d. site × department
site_dept_query = """
SELECT
    site,
    patient_class,
    COUNT(*) AS scans,
    SUM(CASE WHEN radiologist_answer = 'P' THEN 1 ELSE 0 END) AS positives,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS prevalence_pct
FROM filtered_df
GROUP BY site, patient_class
ORDER BY site, patient_class
"""
site_dept_df = duckdb.sql(site_dept_query).df()
display(site_dept_df)

fig_sd = px.bar(
    site_dept_df, x="site", y="scans", color="patient_class", barmode="group", text="prevalence_pct",
    title="Scans & Prevalence (%) by Site and Department",
    labels={"site": "Hospital", "patient_class": "Department"},
    color_discrete_map={"ED": "#e45756", "IN": "#72b7b2"},
)
fig_sd.update_traces(texttemplate="%{text}%", textposition="outside")
fig_sd.update_layout(legend_title_text="")
fig_sd.show()


,site,patient_class,scans,positives,prevalence_pct
0,best_doctors,ED,1649,92.0,5.6
1,best_doctors,IN,4337,505.0,11.6
2,healthy_vibes,ED,3879,169.0,4.4
3,healthy_vibes,IN,10135,1138.0,11.2


---
## 2. When Do Positive Cases Occur?

Positive rate (`radiologist_answer = 'P'`) by hour, day-of-week, and month.  
Radiologist treatment time = minutes from `scan_timestamp` to `radiologist_sign_time`.

In [7]:
# Temporal base table (run this first for all Section 2 cells)
temporal_query = """
SELECT
    EXTRACT(HOUR FROM scan_timestamp) AS hour,
    EXTRACT(DOW FROM scan_timestamp) AS dow,
    EXTRACT(MONTH FROM scan_timestamp) AS month,
    gender,
    CASE WHEN age < 40 THEN 'Young (<40)' WHEN age < 65 THEN 'Middle (40-64)' ELSE 'Senior (65+)' END AS age_group,
    radiologist_answer,
    EPOCH(radiologist_sign_time - scan_timestamp) / 60.0 AS rad_treatment_min
FROM filtered_df
WHERE radiologist_sign_time IS NOT NULL
"""
temporal_df = duckdb.sql(temporal_query).df()
print(f"Temporal rows: {len(temporal_df):,}")


Temporal rows: 20,000


### 2a-i: Hourly Positive Rate by Gender & Age Group

In [8]:
hourly_query = """
SELECT
    hour, gender, age_group,
    COUNT(*) AS scans,
    SUM(CASE WHEN radiologist_answer = 'P' THEN 1 ELSE 0 END) AS positives,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS pos_rate
FROM temporal_df
GROUP BY hour, gender, age_group
ORDER BY hour, gender, age_group
"""
hourly_df = duckdb.sql(hourly_query).df()
display(hourly_df.head(10))

fig_h = px.line(
    hourly_df, x="hour", y="pos_rate", color="age_group", facet_col="gender",
    markers=True,
    title="Hourly Positive Rate by Gender & Age Group",
    labels={"hour": "Hour of Day (0-23)", "pos_rate": "Positive Rate (%)"},
    color_discrete_map={"Young (<40)": "#4c78a8", "Middle (40-64)": "#f58518", "Senior (65+)": "#e45756"},
    category_orders={"age_group": ["Young (<40)", "Middle (40-64)", "Senior (65+)"]},
)
fig_h.update_layout(legend_title_text="")
fig_h.show()


,hour,gender,age_group,scans,positives,pos_rate
0,0,female,Middle (40-64),166,23.0,13.9
1,0,female,Senior (65+),83,5.0,6.0
2,0,female,Young (<40),142,18.0,12.7
3,0,male,Middle (40-64),209,20.0,9.6
4,0,male,Senior (65+),104,8.0,7.7
5,0,male,Young (<40),168,23.0,13.7
6,1,female,Middle (40-64),173,20.0,11.6
7,1,female,Senior (65+),87,9.0,10.3
8,1,female,Young (<40),131,14.0,10.7
9,1,male,Middle (40-64),186,17.0,9.1


### 2a-ii: Hourly Positive Rate & Radiologist Time — Per Gender (all ages)

In [9]:
hourly_gender_query = """
SELECT
    hour, gender,
    COUNT(*) AS scans,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS pos_rate,
    ROUND(AVG(rad_treatment_min), 2) AS mean_rad_min
FROM temporal_df
GROUP BY hour, gender
ORDER BY hour, gender
"""
hourly_gender_df = duckdb.sql(hourly_gender_query).df()
display(hourly_gender_df.head(10))

fig_hg = make_subplots(rows=1, cols=2, subplot_titles=["Female", "Male"], shared_yaxes=True)
for i, g in enumerate(["female", "male"]):
    sub = hourly_gender_df[hourly_gender_df["gender"] == g]
    fig_hg.add_trace(go.Scatter(
        x=sub["hour"], y=sub["pos_rate"], mode="lines+markers",
        name="Positive Rate (%)", line=dict(color="#e45756"), showlegend=(i == 0),
    ), row=1, col=i + 1)
    fig_hg.add_trace(go.Scatter(
        x=sub["hour"], y=sub["mean_rad_min"], mode="lines+markers",
        name="Rad. Treatment (min)", line=dict(color="#2ca02c", dash="dot"), showlegend=(i == 0),
    ), row=1, col=i + 1)
fig_hg.update_yaxes(title_text="Positive Rate (%)", row=1, col=1)
fig_hg.update_layout(title="Hourly Positive Rate & Radiologist Time — Per Gender", showlegend=True)
fig_hg.show()


,hour,gender,scans,pos_rate,mean_rad_min
0,0,female,391,11.8,10.49
1,0,male,481,10.6,10.19
2,1,female,391,11.0,10.34
3,1,male,446,11.4,10.29
4,2,female,379,12.9,10.48
5,2,male,451,10.0,10.23
6,3,female,408,11.8,10.44
7,3,male,460,9.3,10.27
8,4,female,346,10.1,10.31
9,4,male,470,7.7,10.36


### 2a-iii: Overall Hourly Positive Rate vs Radiologist Treatment Time

In [10]:
hourly_overall_query = """
SELECT
    hour,
    COUNT(*) AS scans,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS pos_rate,
    ROUND(AVG(rad_treatment_min), 2) AS mean_rad_min
FROM temporal_df
GROUP BY hour
ORDER BY hour
"""
hourly_overall_df = duckdb.sql(hourly_overall_query).df()
display(hourly_overall_df)

fig_overall = make_subplots(specs=[[{"secondary_y": True}]])
fig_overall.add_trace(go.Scatter(
    x=hourly_overall_df["hour"], y=hourly_overall_df["pos_rate"],
    mode="lines+markers", name="Positive Rate (%)", line=dict(color="#e45756"),
), secondary_y=False)
fig_overall.add_trace(go.Scatter(
    x=hourly_overall_df["hour"], y=hourly_overall_df["mean_rad_min"],
    mode="lines+markers", name="Mean Radiologist Time (min)", line=dict(color="#2ca02c", dash="dot"),
), secondary_y=True)
fig_overall.update_layout(title="Overall: Hourly Positive Rate vs Radiologist Treatment Time")
fig_overall.update_yaxes(title_text="Positive Rate (%)", secondary_y=False)
fig_overall.update_yaxes(title_text="Rad Time (min)", secondary_y=True)
fig_overall.show()


,hour,scans,pos_rate,mean_rad_min
0,0,872,11.1,10.32
1,1,837,11.2,10.32
2,2,830,11.3,10.34
3,3,868,10.5,10.35
4,4,816,8.7,10.34
5,5,808,7.7,10.23
6,6,815,8.1,10.28
7,7,845,10.4,10.34
8,8,829,8.3,10.34
9,9,854,10.8,10.22


### 2a-iv: Correlation — Scan Volume vs Radiologist Time (by Hour)

In [11]:
fig_corr = px.scatter(
    hourly_overall_df, x="scans", y="mean_rad_min",
    text="hour", size="pos_rate",
    title="Correlation: Scan Volume vs Radiologist Time by Hour",
    labels={"scans": "Scans per Hour", "mean_rad_min": "Mean Rad Time (min)"},
)
fig_corr.update_traces(textposition="top center")

corr = np.corrcoef(hourly_overall_df["scans"], hourly_overall_df["mean_rad_min"])[0, 1]
slope, intercept = np.polyfit(hourly_overall_df["scans"], hourly_overall_df["mean_rad_min"], 1)
x_range = [hourly_overall_df["scans"].min(), hourly_overall_df["scans"].max()]
fig_corr.add_trace(go.Scatter(
    x=x_range, y=[slope * x + intercept for x in x_range],
    mode="lines", name=f"Trend (r={corr:.3f})", line=dict(color="grey", dash="dash"),
))
fig_corr.update_layout(showlegend=True)
fig_corr.show()
print(f"Pearson r = {corr:.4f}")


Pearson r = 0.1422


### 2a-v: Total Scan Volume per Hour (all cases, not only positives)

In [12]:
hourly_total_query = """
SELECT
    hour, gender, age_group,
    COUNT(*) AS scans,
    ROUND(AVG(rad_treatment_min), 2) AS mean_rad_min
FROM temporal_df
GROUP BY hour, gender, age_group
ORDER BY hour, gender, age_group
"""
hourly_total_df = duckdb.sql(hourly_total_query).df()
display(hourly_total_df.head(10))

fig_total = px.line(
    hourly_total_df, x="hour", y="scans", color="age_group", facet_col="gender",
    markers=True,
    title="Hourly Total Scan Volume by Gender & Age Group",
    labels={"hour": "Hour of Day (0-23)", "scans": "Total Scans"},
    color_discrete_map={"Young (<40)": "#4c78a8", "Middle (40-64)": "#f58518", "Senior (65+)": "#e45756"},
    category_orders={"age_group": ["Young (<40)", "Middle (40-64)", "Senior (65+)"]},
)
fig_total.update_layout(legend_title_text="")
fig_total.show()


,hour,gender,age_group,scans,mean_rad_min
0,0,female,Middle (40-64),166,10.15
1,0,female,Senior (65+),83,10.86
2,0,female,Young (<40),142,10.68
3,0,male,Middle (40-64),209,10.13
4,0,male,Senior (65+),104,9.93
5,0,male,Young (<40),168,10.42
6,1,female,Middle (40-64),173,10.30
7,1,female,Senior (65+),87,10.08
8,1,female,Young (<40),131,10.58
9,1,male,Middle (40-64),186,10.29


In [13]:
# 2a-v: per gender total scans + radiologist time
hourly_total_gender_query = """
SELECT hour, gender, SUM(scans) AS scans, ROUND(AVG(mean_rad_min), 2) AS mean_rad_min
FROM hourly_total_df
GROUP BY hour, gender
ORDER BY hour, gender
"""
htg_df = duckdb.sql(hourly_total_gender_query).df()
display(htg_df.head(10))

fig_tg = make_subplots(specs=[[{"secondary_y": True}]])
for g, color in [("female", "#f58518"), ("male", "#4c78a8")]:
    sub = htg_df[htg_df["gender"] == g]
    fig_tg.add_trace(go.Scatter(
        x=sub["hour"], y=sub["scans"], mode="lines+markers",
        name=f"{g} scans", line=dict(color=color),
    ), secondary_y=False)
    fig_tg.add_trace(go.Scatter(
        x=sub["hour"], y=sub["mean_rad_min"], mode="lines",
        name=f"{g} rad time", line=dict(color=color, dash="dot"),
    ), secondary_y=True)
fig_tg.update_layout(title="Total Scans & Radiologist Time per Hour — by Gender")
fig_tg.update_yaxes(title_text="Scans", secondary_y=False)
fig_tg.update_yaxes(title_text="Rad Time (min)", secondary_y=True)
fig_tg.show()


,hour,gender,scans,mean_rad_min
0,0,female,391.0,10.56
1,0,male,481.0,10.16
2,1,female,391.0,10.32
3,1,male,446.0,10.30
4,2,female,379.0,10.48
5,2,male,451.0,10.23
6,3,female,408.0,10.40
7,3,male,460.0,10.25
8,4,female,346.0,10.31
9,4,male,470.0,10.37


In [14]:
# 2a-v: overall total scans + radiologist time
hourly_total_overall = htg_df.groupby("hour", as_index=False).agg(
    scans=("scans", "sum"), mean_rad_min=("mean_rad_min", "mean")
)
display(hourly_total_overall)

fig_to = make_subplots(specs=[[{"secondary_y": True}]])
fig_to.add_trace(go.Bar(
    x=hourly_total_overall["hour"], y=hourly_total_overall["scans"],
    name="Total Scans", marker_color="rgba(100,100,100,0.3)",
), secondary_y=False)
fig_to.add_trace(go.Scatter(
    x=hourly_total_overall["hour"], y=hourly_total_overall["mean_rad_min"],
    mode="lines+markers", name="Mean Rad Time (min)", line=dict(color="#2ca02c"),
), secondary_y=True)
fig_to.update_layout(title="Overall Hourly Scan Volume & Radiologist Time")
fig_to.update_yaxes(title_text="Total Scans", secondary_y=False)
fig_to.update_yaxes(title_text="Rad Time (min)", secondary_y=True)
fig_to.show()


,hour,scans,mean_rad_min
0,0,872.0,10.360
1,1,837.0,10.310
2,2,830.0,10.355
3,3,868.0,10.325
4,4,816.0,10.340
5,5,808.0,10.205
6,6,815.0,10.300
7,7,845.0,10.290
8,8,829.0,10.365
9,9,854.0,10.230


In [15]:
# 2b. Day-of-week positive rate
dow_labels = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
dow_query = """
SELECT
    dow, gender, age_group,
    COUNT(*) AS scans,
    SUM(CASE WHEN radiologist_answer = 'P' THEN 1 ELSE 0 END) AS positives,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS pos_rate
FROM temporal_df
GROUP BY dow, gender, age_group
ORDER BY dow, gender, age_group
"""
dow_df = duckdb.sql(dow_query).df()
dow_df["day_name"] = dow_df["dow"].map(dow_labels)
display(dow_df.head(10))

fig_d = px.bar(
    dow_df, x="day_name", y="pos_rate", color="age_group", facet_col="gender", barmode="group",
    title="Weekly Positive Rate by Gender & Age Group",
    labels={"day_name": "Day of Week", "pos_rate": "Positive Rate (%)"},
    color_discrete_map={"Young (<40)": "#4c78a8", "Middle (40-64)": "#f58518", "Senior (65+)": "#e45756"},
    category_orders={"day_name": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]},
)
fig_d.update_layout(legend_title_text="")
fig_d.show()


,dow,gender,age_group,scans,positives,pos_rate,day_name
0,0,female,Middle (40-64),623,50.0,8.0,Mon
1,0,female,Senior (65+),313,20.0,6.4,Mon
2,0,female,Young (<40),442,45.0,10.2,Mon
3,0,male,Middle (40-64),678,62.0,9.1,Mon
4,0,male,Senior (65+),360,22.0,6.1,Mon
5,0,male,Young (<40),462,45.0,9.7,Mon
6,1,female,Middle (40-64),624,74.0,11.9,Tue
7,1,female,Senior (65+),244,18.0,7.4,Tue
8,1,female,Young (<40),450,54.0,12.0,Tue
9,1,male,Middle (40-64),712,82.0,11.5,Tue


In [16]:
# 2c. Monthly positive rate
month_labels = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
monthly_query = """
SELECT
    month, gender, age_group,
    COUNT(*) AS scans,
    SUM(CASE WHEN radiologist_answer = 'P' THEN 1 ELSE 0 END) AS positives,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS pos_rate
FROM temporal_df
GROUP BY month, gender, age_group
ORDER BY month, gender, age_group
"""
monthly_df = duckdb.sql(monthly_query).df()
monthly_df["month_name"] = monthly_df["month"].map(month_labels)
display(monthly_df.head(10))

fig_m = px.line(
    monthly_df, x="month_name", y="pos_rate", color="age_group", facet_col="gender",
    markers=True,
    title="Monthly Positive Rate by Gender & Age Group",
    labels={"month_name": "Month", "pos_rate": "Positive Rate (%)"},
    color_discrete_map={"Young (<40)": "#4c78a8", "Middle (40-64)": "#f58518", "Senior (65+)": "#e45756"},
    category_orders={"month_name": ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]},
)
fig_m.update_layout(legend_title_text="")
fig_m.show()


,month,gender,age_group,scans,positives,pos_rate,month_name
0,1,female,Middle (40-64),375,25.0,6.7,Jan
1,1,female,Senior (65+),157,13.0,8.3,Jan
2,1,female,Young (<40),286,36.0,12.6,Jan
3,1,male,Middle (40-64),436,50.0,11.5,Jan
4,1,male,Senior (65+),229,15.0,6.6,Jan
5,1,male,Young (<40),312,33.0,10.6,Jan
6,2,female,Middle (40-64),308,25.0,8.1,Feb
7,2,female,Senior (65+),169,9.0,5.3,Feb
8,2,female,Young (<40),247,32.0,13.0,Feb
9,2,male,Middle (40-64),400,31.0,7.8,Feb


In [17]:
# 2d. Hour × day-of-week heatmap
hour_dow_query = """
SELECT
    CAST(dow AS INTEGER) AS dow,
    CAST(hour AS INTEGER) AS hour,
    COUNT(*) AS scans,
    ROUND(SUM(CASE WHEN radiologist_answer = 'P' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS pos_rate
FROM temporal_df
GROUP BY dow, hour
ORDER BY dow, hour
"""
hd_df = duckdb.sql(hour_dow_query).df()
dow_labels_list = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
hd_df["day_name"] = hd_df["dow"].map(dict(enumerate(dow_labels_list)))
pivot = hd_df.pivot(index="day_name", columns="hour", values="pos_rate").reindex(dow_labels_list)
display(pivot.head())

fig_heat = px.imshow(
    pivot, aspect="auto",
    title="Positive Rate (%) Heatmap: Day-of-Week × Hour",
    labels={"x": "Hour", "y": "Day", "color": "Pos Rate (%)"},
    color_continuous_scale="YlOrRd",
)
fig_heat.show()


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
day_name,,,,,,,,,,,,,,,,,,,,,
Mon,9.8,6.8,12.9,12.8,4.8,7.6,9.2,12.8,7.9,7.8,...,8.2,6.1,13.0,8.0,9.1,8.5,8.5,10.5,8.1,5.4
Tue,6.9,8.0,13.7,12.7,12.8,10.1,7.7,11.4,5.9,10.0,...,14.8,14.2,8.7,13.2,11.0,8.5,3.1,7.8,7.0,12.3
Wed,13.4,14.5,7.7,8.6,8.5,5.7,6.4,6.4,13.4,10.4,...,7.1,6.8,9.1,11.6,9.5,7.6,10.6,8.3,10.9,9.6
Thu,13.2,14.9,18.4,9.0,6.1,7.8,11.4,10.6,6.2,11.1,...,10.5,10.2,6.3,5.8,8.1,13.8,8.6,12.0,10.0,11.4
Fri,7.0,11.6,9.2,12.5,13.3,8.0,7.3,11.7,7.4,11.6,...,12.3,13.3,8.9,9.1,12.0,10.4,5.5,7.6,8.5,9.9


---
## 3. Age Statistics (Median, Mean, Min, Max)

Age stats by hospital site and department (`ED`, `IN`).

In [18]:
# 3a. per hospital
age_site_query = """
SELECT
    site AS hospital,
    COUNT(*) AS scans,
    ROUND(AVG(age), 1) AS mean_age,
    ROUND(MEDIAN(age), 1) AS median_age,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    ROUND(STDDEV(age), 1) AS std_age
FROM filtered_df
GROUP BY site
ORDER BY site
"""
age_site_df = duckdb.sql(age_site_query).df()
display(age_site_df)

fig_box_site = px.box(
    filtered_df, x="site", y="age", color="site",
    title="Age Distribution by Hospital",
    labels={"site": "Hospital", "age": "Age (years)"},
    color_discrete_map={"best_doctors": "#4c78a8", "healthy_vibes": "#f58518"},
    points=False,
)
fig_box_site.update_layout(showlegend=False, xaxis_title="")
fig_box_site.show()


,hospital,scans,mean_age,median_age,min_age,max_age,std_age
0,best_doctors,5986,48.1,49.0,6,86,18.1
1,healthy_vibes,14014,48.6,49.0,5,86,17.8


In [19]:
# 3b. per department
age_dept_query = """
SELECT
    patient_class AS department,
    COUNT(*) AS scans,
    ROUND(AVG(age), 1) AS mean_age,
    ROUND(MEDIAN(age), 1) AS median_age,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    ROUND(STDDEV(age), 1) AS std_age
FROM filtered_df
GROUP BY patient_class
ORDER BY patient_class
"""
age_dept_df = duckdb.sql(age_dept_query).df()
display(age_dept_df)

fig_box_dept = px.box(
    filtered_df, x="patient_class", y="age", color="patient_class",
    title="Age Distribution by Department",
    labels={"patient_class": "Department", "age": "Age (years)"},
    color_discrete_map={"ED": "#e45756", "IN": "#72b7b2"},
    points=False,
)
fig_box_dept.update_layout(showlegend=False, xaxis_title="")
fig_box_dept.show()


,department,scans,mean_age,median_age,min_age,max_age,std_age
0,ED,5528,48.7,49.0,6,86,18.0
1,IN,14472,48.4,49.0,5,86,17.9


In [20]:
# 3c. site × department × gender
age_full_query = """
SELECT
    site AS hospital,
    patient_class AS department,
    gender,
    COUNT(*) AS scans,
    ROUND(AVG(age), 1) AS mean_age,
    ROUND(MEDIAN(age), 1) AS median_age,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    ROUND(STDDEV(age), 1) AS std_age
FROM filtered_df
GROUP BY site, patient_class, gender
ORDER BY site, patient_class, gender
"""
age_full_df = duckdb.sql(age_full_query).df()
display(age_full_df)

fig_violin = px.violin(
    filtered_df, x="patient_class", y="age", color="gender",
    facet_col="site", box=True, points=False,
    title="Age Distribution: Site × Department × Gender",
    labels={"patient_class": "Department", "age": "Age (years)"},
    color_discrete_map={"male": "#4c78a8", "female": "#f58518"},
)
fig_violin.update_layout(legend_title_text="")
fig_violin.show()


,hospital,department,gender,scans,mean_age,median_age,min_age,max_age,std_age
0,best_doctors,ED,female,602,49.3,49.0,6,86,17.7
1,best_doctors,ED,male,1047,47.2,47.0,6,86,18.8
2,best_doctors,IN,female,2221,48.1,49.0,7,85,18.1
3,best_doctors,IN,male,2116,48.3,49.0,6,86,17.9
4,healthy_vibes,ED,female,1377,49.0,49.0,7,86,17.6
5,healthy_vibes,ED,male,2502,49.0,49.0,6,86,17.8
6,healthy_vibes,IN,female,5020,48.6,49.0,5,85,17.7
7,healthy_vibes,IN,male,5115,48.4,49.0,5,86,17.9


---
## 4. Algorithm Accuracy by Patient Class

In [21]:
pc_df = duckdb.sql(group_accuracy_query("patient_class")).df()
display(pc_df)
accuracy_bar(pc_df, "patient_class", "Accuracy (%) by Patient Class").show()


,patient_class,scans,positives,prevalence_pct,Algo 1,Algo 2,Algo 3
0,ED,5528,261.0,4.7,96.5,45.4,96.0
1,IN,14472,1643.0,11.4,92.7,48.3,94.4


## 5. Algorithm Accuracy by Gender

In [22]:
gender_df = duckdb.sql(group_accuracy_query("gender")).df()
display(gender_df)
accuracy_bar(gender_df, "gender", "Accuracy (%) by Gender").show()


,gender,scans,positives,prevalence_pct,Algo 1,Algo 2,Algo 3
0,female,9220,898.0,9.7,93.6,46.9,95.0
1,male,10780,1006.0,9.3,93.9,47.9,94.7


## 6. Age Subgroups

Radiologist line = positive rate (prevalence). Grey bars = scan count per bin.

In [23]:
age_df = duckdb.sql(AGE_QUERY).df()
display(age_df)
age_subgroup_chart(age_df).show()


,age_group,scans,Radiologist,Algo 1,Algo 2,Algo 3
0,03-06,4,0.0,100.0,50.0,100.0
1,06-09,83,6.0,95.2,67.5,98.8
2,09-12,176,10.2,93.2,61.9,93.8
3,12-15,301,12.0,91.0,66.8,93.0
4,15-18,381,12.9,93.7,61.4,94.8
5,18-21,419,12.2,93.8,59.9,92.8
6,21-24,546,13.2,91.8,61.7,94.7
7,24-27,668,9.9,93.4,60.6,95.1
8,27-30,704,11.4,92.6,60.5,93.9
9,30-32,580,9.7,94.5,63.1,93.6


## 7. Timelines (TAT)

Minutes from `algos_start_run`. Negative durations excluded.

In [24]:
duration_query = """
WITH duration_calc AS (
    SELECT
        EPOCH(radiologist_sign_time - algos_start_run) / 60.0 AS rad_min,
        EPOCH(algo1_finish_run - algos_start_run) / 60.0 AS algo1_min,
        EPOCH(algo2_finish_run - algos_start_run) / 60.0 AS algo2_min,
        EPOCH(algo3_finish_run - algos_start_run) / 60.0 AS algo3_min
    FROM filtered_df
    WHERE radiologist_sign_time IS NOT NULL
      AND algos_start_run IS NOT NULL
      AND algo1_finish_run IS NOT NULL
      AND algo2_finish_run IS NOT NULL
      AND algo3_finish_run IS NOT NULL
)
SELECT rad_min AS "Radiologist", algo1_min AS "Algo 1", algo2_min AS "Algo 2", algo3_min AS "Algo 3"
FROM duration_calc
WHERE rad_min >= 0 AND algo1_min >= 0 AND algo2_min >= 0 AND algo3_min >= 0
"""
duration_df = duckdb.sql(duration_query).df()
timeline_df = pd.DataFrame({
    "Entity": duration_df.columns,
    "Mean (min)": duration_df.mean().round(2).values,
    "Median (min)": duration_df.median().round(2).values,
    "Min (min)": duration_df.min().round(2).values,
    "Max (min)": duration_df.max().round(2).values,
})
display(timeline_df)
tat_long = duration_df.melt(var_name="Entity", value_name="Minutes")
px.box(
    tat_long, x="Entity", y="Minutes", color="Entity",
    color_discrete_map=COLOR_MAP, title="TAT Distribution (minutes)", points=False,
).update_layout(showlegend=False, xaxis_title="").show()


,Entity,Mean (min),Median (min),Min (min),Max (min)
0,Radiologist,9.73,9.65,0.32,19.56
1,Algo 1,0.75,0.75,0.00,1.72
2,Algo 2,2.50,2.50,0.47,4.59
3,Algo 3,10.02,10.03,2.44,18.06


## 8. Clinical Metrics

In [25]:
metrics_query = """
WITH cm AS (
    SELECT 'Algo 1' AS Algorithm,
        SUM(CASE WHEN algo1_answer = 'P' AND radiologist_answer = 'P' THEN 1 ELSE 0 END) AS TP,
        SUM(CASE WHEN algo1_answer = 'N' AND radiologist_answer = 'N' THEN 1 ELSE 0 END) AS TN,
        SUM(CASE WHEN algo1_answer = 'P' AND radiologist_answer = 'N' THEN 1 ELSE 0 END) AS FP,
        SUM(CASE WHEN algo1_answer = 'N' AND radiologist_answer = 'P' THEN 1 ELSE 0 END) AS FN
    FROM filtered_df
    UNION ALL
    SELECT 'Algo 2',
        SUM(CASE WHEN algo2_answer = 'P' AND radiologist_answer = 'P' THEN 1 ELSE 0 END),
        SUM(CASE WHEN algo2_answer = 'N' AND radiologist_answer = 'N' THEN 1 ELSE 0 END),
        SUM(CASE WHEN algo2_answer = 'P' AND radiologist_answer = 'N' THEN 1 ELSE 0 END),
        SUM(CASE WHEN algo2_answer = 'N' AND radiologist_answer = 'P' THEN 1 ELSE 0 END)
    FROM filtered_df
    UNION ALL
    SELECT 'Algo 3',
        SUM(CASE WHEN algo3_answer = 'P' AND radiologist_answer = 'P' THEN 1 ELSE 0 END),
        SUM(CASE WHEN algo3_answer = 'N' AND radiologist_answer = 'N' THEN 1 ELSE 0 END),
        SUM(CASE WHEN algo3_answer = 'P' AND radiologist_answer = 'N' THEN 1 ELSE 0 END),
        SUM(CASE WHEN algo3_answer = 'N' AND radiologist_answer = 'P' THEN 1 ELSE 0 END)
    FROM filtered_df
)
SELECT Algorithm, TP, TN, FP, FN,
    ROUND((TP * 100.0) / NULLIF(TP + FN, 0), 2) AS "Sensitivity (%)",
    ROUND((TN * 100.0) / NULLIF(TN + FP, 0), 2) AS "Specificity (%)",
    ROUND((TP * 100.0) / NULLIF(TP + FP, 0), 2) AS "Precision (%)",
    ROUND(((TP + TN) * 100.0) / NULLIF(TP + TN + FP + FN, 0), 2) AS "Accuracy (%)"
FROM cm
"""
metrics_df = duckdb.sql(metrics_query).df()
display(metrics_df)
radar_metrics = ["Sensitivity (%)", "Specificity (%)", "Precision (%)", "Accuracy (%)"]
fig_radar = go.Figure()
for _, row in metrics_df.iterrows():
    values = [row[m] for m in radar_metrics]
    fig_radar.add_trace(go.Scatterpolar(
        r=values + [values[0]], theta=radar_metrics + [radar_metrics[0]],
        name=row["Algorithm"], line=dict(color=COLOR_MAP[row["Algorithm"]]),
        fill="toself", fillcolor=COLOR_MAP[row["Algorithm"]], opacity=0.45,
    ))
fig_radar.update_layout(title="Diagnostic Profile", polar=dict(radialaxis=dict(visible=True, range=[0, 100])))
fig_radar.show()


,Algorithm,TP,TN,FP,FN,Sensitivity (%),Specificity (%),Precision (%),Accuracy (%)
0,Algo 1,965.0,17784.0,312.0,939.0,50.68,98.28,75.57,93.75
1,Algo 2,1743.0,7749.0,10347.0,161.0,91.54,42.82,14.42,47.46
2,Algo 3,1403.0,17559.0,537.0,501.0,73.69,97.03,72.32,94.81


## 9. Site Accuracy

In [26]:
site_query = """
SELECT site, COUNT(*) AS scans,
    ROUND(AVG(CASE WHEN algo1_answer = radiologist_answer THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Algo 1",
    ROUND(AVG(CASE WHEN algo2_answer = radiologist_answer THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Algo 2",
    ROUND(AVG(CASE WHEN algo3_answer = radiologist_answer THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Algo 3"
FROM filtered_df
GROUP BY site
ORDER BY site
"""
site_acc = duckdb.sql(site_query).df()
display(site_acc)
site_long = site_acc.melt(id_vars="site", value_vars=ALGO_COLS, var_name="Algorithm", value_name="Accuracy (%)")
px.bar(
    site_long, x="site", y="Accuracy (%)", color="Algorithm", barmode="group",
    title="Accuracy (%) by Hospital Site", color_discrete_map=COLOR_MAP,
    labels={"site": "Hospital Site"},
).update_layout(legend_title_text="").show()


,site,scans,Algo 1,Algo 2,Algo 3
0,best_doctors,5986,93.8,47.6,94.7
1,healthy_vibes,14014,93.7,47.4,94.9
